In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


torch.manual_seed(0)
n = 100
x = torch.rand(n, 1)
y = torch.sin(2 * torch.pi * x) + torch.rand(n, 1)

In [19]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self, input_size=1, hidden_size=10, output_size=1):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        y = self.linear1(x)
        y = F.sigmoid(y)
        y = self.linear2(y)
        return y

# 隠れ層の数を指定可能にする
class Model2(nn.Module):
    def __init__(self, input_size=1, hidden_size=10, output_size=1, num_hidden_layers=1):
        super().__init__()
        layers = []
        layers.append(nn.Linear(input_size, hidden_size))
        for _ in range(num_hidden_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
        self.hidden_layers = nn.ModuleList(layers)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        x = self.output_layer(x)
        return x


In [ ]:
import csv
import os
import numpy as np

lr = 0.1
iters = 10000
hidden_size = 10
results_dir = os.path.join('results', 'nb06c')
png_dir = os.path.join(results_dir, 'png')
gif_dir = os.path.join(results_dir, 'gif')
os.makedirs(results_dir, exist_ok=True)
os.makedirs(png_dir, exist_ok=True)
os.makedirs(gif_dir, exist_ok=True)

# アクティベーション記録用の入力データ
torch.manual_seed(0)
input_data = torch.rand(1000, 1)

# アクティベーション保存間隔（itersを10分割）
act_save_interval = iters // 10

　

In [ ]:
## 各隠れ層の活性化（ReLU通過後）のヒストグラムを描画（初期重み）
## weight_init_activation_histogram.py のスタイル

import numpy as np
import math

results_dir = os.path.join('results', 'nb06c')
png_dir = os.path.join(results_dir, 'png')

torch.manual_seed(0)
input_data = torch.rand(1000, 1)  # toy dataset と同じ入力次元(1)

for num_hidden in [2, 20]:
    hidden_size = 10
    model = Model2(hidden_size=hidden_size, num_hidden_layers=num_hidden)
    
    # forward しながら各隠れ層の activation を記録
    activations = {}
    h = input_data
    for i, layer in enumerate(model.hidden_layers):
        h = F.relu(layer(h))
        activations[i] = h.detach().numpy()
    
    # CSV保存: 各層の全ニューロンのraw activation値
    csv_path = os.path.join(results_dir, f'06c_activations_hidden{num_hidden}_hidSz{hidden_size}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['num_hidden_layers', 'hidden_size', 'layer', 'neuron_index', 'value'])
        for layer_idx, act in activations.items():
            # act shape: (1000, hidden_size)
            for neuron_idx in range(act.shape[1]):
                for val in act[:, neuron_idx]:
                    writer.writerow([num_hidden, hidden_size, layer_idx, neuron_idx, val])
    print(f'saved: {csv_path}')
    
    # ヒストグラム描画（最大4列のグリッド配置）
    n_layers = len(activations)
    max_cols = 4
    ncols = min(n_layers, max_cols)
    nrows = math.ceil(n_layers / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.5))
    if n_layers == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    fig.suptitle(f'Activation histogram (num_hidden_layers={num_hidden}, hidden_size={hidden_size})\nInitial weights (before training)', fontsize=12)
    
    for i in range(n_layers):
        ax = axes[i]
        data = activations[i].flatten()
        ax.hist(data, 30, range=(0, data.max() + 0.01) if data.max() > 0 else (0, 1))
        ax.set_title(f'Layer {i+1}')
        # 0の割合を表示
        zero_ratio = np.sum(data == 0) / len(data) * 100
        ax.text(0.95, 0.95, f'{zero_ratio:.0f}% dead', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, color='red')
    
    # 余ったサブプロットを非表示に
    for j in range(n_layers, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    png_path = os.path.join(png_dir, f'06c_act_init_hidden{num_hidden}.png')
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    print(f'saved: {png_path}')
    plt.show()

In [ ]:
# relu
print("loss(relu): ")

num_hidden_layers_list = [2, 3, 5, 10, 20]  # 隠れ層の数を指定するリスト

for num_hidden_layers in num_hidden_layers_list:
    model2 = Model2(hidden_size=hidden_size, num_hidden_layers=num_hidden_layers)
    optimizer2 = torch.optim.SGD(model2.parameters(), lr=lr)

    # CSV保存用
    csv_path = os.path.join(results_dir, f'06c_loss_hidden{num_hidden_layers}_hidSz{hidden_size}_lr{lr}.csv')
    loss_records = []

    # アクティベーション遷移CSV保存用
    act_csv_path = os.path.join(results_dir, f'06c_act_training_hidden{num_hidden_layers}_hidSz{hidden_size}.csv')
    act_records = []

    for i in range(iters):
        y_pred2 = model2(x)
        loss2 = F.mse_loss(y, y_pred2)
        
        loss2.backward()
        optimizer2.step()
        optimizer2.zero_grad()
        
        if i % 1000 == 0:
            print(loss2.item())
            loss_records.append([num_hidden_layers, lr, hidden_size, i, loss2.item()])

        # アクティベーションを10分割のタイミングで記録
        if i % act_save_interval == 0 or i == iters - 1:
            model2.eval()
            with torch.no_grad():
                h = input_data
                for layer_idx, layer in enumerate(model2.hidden_layers):
                    h = F.relu(layer(h))
                    # 各ニューロンの値を記録
                    act_np = h.numpy()
                    for neuron_idx in range(act_np.shape[1]):
                        for val in act_np[:, neuron_idx]:
                            act_records.append([num_hidden_layers, hidden_size, i, layer_idx, neuron_idx, val])
            model2.train()

    # 最終iterationのlossも記録（最後のi%1000!=0の場合）
    if (iters - 1) % 1000 != 0:
        loss_records.append([num_hidden_layers, lr, hidden_size, iters - 1, loss2.item()])

    # Loss CSV書き出し
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['num_hidden_layers', 'lr', 'hidden_size', 'iteration', 'loss'])
        writer.writerows(loss_records)
    print(f'  -> saved: {csv_path}')

    # アクティベーション遷移CSV書き出し
    with open(act_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['num_hidden_layers', 'hidden_size', 'iteration', 'layer', 'neuron_index', 'value'])
        writer.writerows(act_records)
    print(f'  -> saved: {act_csv_path}')

    # 学習後のモデルの出力を可視化
    plt.scatter(x.numpy(), y.numpy(), s=10, label='training data')
    x_test2 = torch.linspace(0, 1, 300).reshape(-1, 1)
    model2.eval()
    with torch.no_grad():
        y_test2 = model2(x_test2)

    plt.plot(x_test2.numpy(), y_test2.numpy(), label=f'by relu (hidden layers: {len(model2.hidden_layers)})', color = 'blue')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Toy Dataset (n={n})\n y = sin(2πx) + ε')
    plt.legend()
    png_path = os.path.join(png_dir, f'06c_fit_hidden{num_hidden_layers}.png')
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    print(f'  -> saved: {png_path}')
    plt.show()

In [ ]:
## CSVから読み込んで分析

import csv
import glob as glob_mod
import numpy as np
import math

results_dir = os.path.join('results', 'nb06c')
png_dir = os.path.join(results_dir, 'png')

# --- 1. 学習曲線比較: 全 num_hidden_layers の loss 推移を1枚に重ねてプロット ---
loss_files = sorted(glob_mod.glob(os.path.join(results_dir, '06c_loss_hidden*.csv')))
print(f'Loss CSVs found: {len(loss_files)}')

plt.figure(figsize=(8, 5))
for fpath in loss_files:
    with open(fpath, newline='') as f:
        reader = csv.DictReader(f)
        iterations = []
        losses = []
        num_hidden_layers_val = None
        for row in reader:
            if num_hidden_layers_val is None:
                num_hidden_layers_val = row['num_hidden_layers']
            iterations.append(float(row['iteration']))
            losses.append(float(row['loss']))
    label = f'hidden_layers={num_hidden_layers_val}'
    plt.plot(iterations, losses, marker='o', markersize=3, label=label)

plt.xlabel('Iteration')
plt.ylabel('Loss (MSE)')
plt.title('Training Loss Comparison by Number of Hidden Layers')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
png_path = os.path.join(png_dir, '06c_loss_comparison.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight')
print(f'saved: {png_path}')
plt.show()

# --- 2. 初期重みの活性化ヒストグラム再描画（最大4列） ---
act_files = sorted(glob_mod.glob(os.path.join(results_dir, '06c_activations_hidden*.csv')))
print(f'Activation CSVs (initial weights) found: {len(act_files)}')

for fpath in act_files:
    with open(fpath, newline='') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    num_hidden = rows[0]['num_hidden_layers']
    hidden_size = rows[0]['hidden_size']

    # layer ごとに value を集める
    layer_values = {}
    for row in rows:
        layer_id = int(row['layer'])
        val = float(row['value'])
        layer_values.setdefault(layer_id, []).append(val)

    layers = sorted(layer_values.keys())
    n_layers = len(layers)

    max_cols = 4
    ncols = min(n_layers, max_cols)
    nrows = math.ceil(n_layers / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.5))
    if n_layers == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    fig.suptitle(f'Activation histogram (num_hidden_layers={num_hidden}, hidden_size={hidden_size})\n[from CSV, initial weights]', fontsize=12)

    for idx, layer_id in enumerate(layers):
        ax = axes[idx]
        data = np.array(layer_values[layer_id])
        ax.hist(data, 30, range=(0, data.max() + 0.01) if data.max() > 0 else (0, 1))
        ax.set_title(f'Layer {layer_id + 1}')
        zero_ratio = np.sum(data == 0) / len(data) * 100
        ax.text(0.95, 0.95, f'{zero_ratio:.0f}% dead', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, color='red')

    for j in range(n_layers, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    png_path = os.path.join(png_dir, f'06c_act_init_csv_hidden{num_hidden}.png')
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    print(f'saved: {png_path}')
    plt.show()

# --- 3. 学習中のアクティベーション遷移 ---
act_training_files = sorted(glob_mod.glob(os.path.join(results_dir, '06c_act_training_hidden*.csv')))
print(f'Activation training CSVs found: {len(act_training_files)}')

for fpath in act_training_files:
    with open(fpath, newline='') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        continue

    num_hidden = rows[0]['num_hidden_layers']
    hidden_size_val = rows[0]['hidden_size']

    # iteration x layer ごとに value を集める
    data_by_iter_layer = {}
    for row in rows:
        iteration = int(row['iteration'])
        layer_id = int(row['layer'])
        val = float(row['value'])
        data_by_iter_layer.setdefault((iteration, layer_id), []).append(val)

    iterations_set = sorted(set(k[0] for k in data_by_iter_layer.keys()))
    layers_set = sorted(set(k[1] for k in data_by_iter_layer.keys()))
    n_layers = len(layers_set)

    # --- 3a. 各層のdead neuron割合の推移（折れ線グラフ） ---
    plt.figure(figsize=(8, 5))
    for layer_id in layers_set:
        dead_ratios = []
        for it in iterations_set:
            vals = np.array(data_by_iter_layer.get((it, layer_id), []))
            dead_ratio = np.sum(vals == 0) / len(vals) * 100 if len(vals) > 0 else 0
            dead_ratios.append(dead_ratio)
        plt.plot(iterations_set, dead_ratios, marker='o', markersize=3, label=f'Layer {layer_id + 1}')

    plt.xlabel('Iteration')
    plt.ylabel('Dead neuron ratio (%)')
    plt.title(f'Dead Neuron Ratio During Training\n(num_hidden_layers={num_hidden}, hidden_size={hidden_size_val})')
    plt.legend(fontsize=7, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    png_path = os.path.join(png_dir, f'06c_dead_ratio_hidden{num_hidden}.png')
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    print(f'saved: {png_path}')
    plt.show()

    # --- 3b. 各層の活性化の標準偏差の推移 ---
    plt.figure(figsize=(8, 5))
    for layer_id in layers_set:
        stds = []
        for it in iterations_set:
            vals = np.array(data_by_iter_layer.get((it, layer_id), []))
            stds.append(np.std(vals) if len(vals) > 0 else 0)
        plt.plot(iterations_set, stds, marker='o', markersize=3, label=f'Layer {layer_id + 1}')

    plt.xlabel('Iteration')
    plt.ylabel('Activation std')
    plt.title(f'Activation Std During Training\n(num_hidden_layers={num_hidden}, hidden_size={hidden_size_val})')
    plt.legend(fontsize=7, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    png_path = os.path.join(png_dir, f'06c_act_std_hidden{num_hidden}.png')
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    print(f'saved: {png_path}')
    plt.show()

    # --- 3c. スナップショットヒストグラム: 代表的なiteration時点でのヒストグラム ---
    # 最初・中間・最後の3時点を選択
    snap_iters = [iterations_set[0], iterations_set[len(iterations_set)//2], iterations_set[-1]]
    for snap_it in snap_iters:
        max_cols = 4
        ncols = min(n_layers, max_cols)
        nrows_grid = math.ceil(n_layers / ncols)
        fig, axes = plt.subplots(nrows_grid, ncols, figsize=(ncols * 3, nrows_grid * 2.5))
        if n_layers == 1:
            axes = np.array([axes])
        axes = np.array(axes).flatten()
        fig.suptitle(f'Activation histogram at iter={snap_it}\n(num_hidden={num_hidden}, hidden_size={hidden_size_val})', fontsize=12)

        for idx, layer_id in enumerate(layers_set):
            ax = axes[idx]
            data = np.array(data_by_iter_layer.get((snap_it, layer_id), []))
            ax.hist(data, 30, range=(0, data.max() + 0.01) if len(data) > 0 and data.max() > 0 else (0, 1))
            ax.set_title(f'Layer {layer_id + 1}')
            zero_ratio = np.sum(data == 0) / len(data) * 100 if len(data) > 0 else 0
            ax.text(0.95, 0.95, f'{zero_ratio:.0f}% dead', transform=ax.transAxes,
                    ha='right', va='top', fontsize=8, color='red')

        for j in range(n_layers, len(axes)):
            axes[j].set_visible(False)

        plt.tight_layout()
        png_path = os.path.join(png_dir, f'06c_act_snapshot_hidden{num_hidden}_iter{snap_it}.png')
        plt.savefig(png_path, dpi=150, bbox_inches='tight')
        print(f'saved: {png_path}')
        plt.show()

In [ ]:
## アクティベーション遷移のGIFアニメーション生成
## 各イテレーション時点での全層ヒストグラムを1フレームとし、学習の進行をアニメーション化

import csv
import glob as glob_mod
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

results_dir = os.path.join('results', 'nb06c')
gif_dir = os.path.join(results_dir, 'gif')
act_training_files = sorted(glob_mod.glob(os.path.join(results_dir, '06c_act_training_hidden*.csv')))
print(f'Activation training CSVs found: {len(act_training_files)}')

for fpath in act_training_files:
    with open(fpath, newline='') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        continue

    num_hidden = rows[0]['num_hidden_layers']
    hidden_size_val = rows[0]['hidden_size']

    # iteration x layer ごとに value を集める
    data_by_iter_layer = {}
    for row in rows:
        iteration = int(row['iteration'])
        layer_id = int(row['layer'])
        val = float(row['value'])
        data_by_iter_layer.setdefault((iteration, layer_id), []).append(val)

    iterations_set = sorted(set(k[0] for k in data_by_iter_layer.keys()))
    layers_set = sorted(set(k[1] for k in data_by_iter_layer.keys()))
    n_layers = len(layers_set)

    # 全フレーム共通のヒストグラム範囲を決定
    all_vals = np.array([float(r['value']) for r in rows])
    hist_max = np.percentile(all_vals[all_vals > 0], 99) if np.any(all_vals > 0) else 1.0
    hist_range = (0, hist_max)

    # 全フレーム共通のy軸範囲を事前計算
    global_y_max = 0
    for it in iterations_set:
        for layer_id in layers_set:
            data = np.array(data_by_iter_layer.get((it, layer_id), []))
            counts, _ = np.histogram(data, bins=30, range=hist_range)
            global_y_max = max(global_y_max, counts.max())

    # グリッドレイアウト（最大4列）
    max_cols = 4
    ncols = min(n_layers, max_cols)
    nrows = math.ceil(n_layers / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.5))
    if n_layers == 1:
        axes = np.array([axes])
    axes_flat = np.array(axes).flatten()

    # 余ったサブプロットを非表示に
    for j in range(n_layers, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle('', fontsize=12)

    def update(frame_idx):
        it = iterations_set[frame_idx]
        fig.suptitle(
            f'Activation histogram (iter={it})\n'
            f'(num_hidden={num_hidden}, hidden_size={hidden_size_val})',
            fontsize=12
        )
        for idx, layer_id in enumerate(layers_set):
            ax = axes_flat[idx]
            ax.clear()
            data = np.array(data_by_iter_layer.get((it, layer_id), []))
            ax.hist(data, 30, range=hist_range)
            ax.set_title(f'Layer {layer_id + 1}')
            ax.set_ylim(0, global_y_max * 1.1)
            zero_ratio = np.sum(data == 0) / len(data) * 100 if len(data) > 0 else 0
            ax.text(0.95, 0.95, f'{zero_ratio:.0f}% dead', transform=ax.transAxes,
                    ha='right', va='top', fontsize=8, color='red')
        fig.tight_layout(rect=[0, 0, 1, 0.92])
        return axes_flat[:n_layers]

    anim = FuncAnimation(fig, update, frames=len(iterations_set), interval=800, blit=False)

    gif_path = os.path.join(gif_dir, f'06c_act_evolution_hidden{num_hidden}.gif')
    anim.save(gif_path, writer=PillowWriter(fps=2))
    plt.close(fig)
    print(f'saved: {gif_path}')
    display(Image(filename=gif_path))